# Corpus mixtures: equal hours and equal-update controls

Mixture construction and training are now reproducible from shared code and frozen manifests. The main new control compares approximately 64.32h of Neyshekar-only training against 64.32h of mixed training. A second grid holds optimizer updates equal as well.

In [ ]:
# Resolve shared code when launched from code/ or the repository root.
import sys
from pathlib import Path

_code_candidates = [Path.cwd(), Path.cwd() / "code"]
CODE = next(
    (
        p.resolve()
        for p in _code_candidates
        if (p / "neyshekar_experiments" / "protocol.py").is_file()
    ),
    None,
)
if CODE is None:
    raise RuntimeError("Launch this notebook from the repository root or its code/ directory.")
if str(CODE) not in sys.path:
    sys.path.insert(0, str(CODE))
from neyshekar_experiments.protocol import ROOT
# End notebook bootstrap

import pandas as pd
from IPython.display import display
from neyshekar_experiments.manifests import prepare
from neyshekar_experiments.training import experiment_grid, plan, execute
from neyshekar_experiments.reporting import family_scores, corrected_results, metric_figure

# Imports and reports never start training; paths use the shared repository root.

RUN_TRAINING = False
DATA_SEED = 42  # All manifests are frozen with this seed; optimization seeds are separate.

# Create/verify manifests from source data when this notebook is run.
manifest_summary = prepare()

## 1. Data budgets

The 32h mixture contains approximately 16h from each corpus; the 64h mixture contains the two complete 32h conditions. Sampling and source proportions are recorded in each manifest.

In [ ]:
from neyshekar_experiments.manifests import load_manifest

names = ["ney_matched", "cv_matched", "mixed_matched", "mixed_double", "ney_double"]
display(
    pd.DataFrame(
        [
            {k: load_manifest(name)[k] for k in ("name", "clips", "hours", "source_hours")}
            for name in names
        ]
    )
)

## 2. Fresh WER and CER

Evaluate equal-hour and equal-update conditions separately. Missing comparisons stay pending.

In [ ]:
fresh = family_scores("mixture")
display(fresh)
if not fresh.empty:
    for _, group in fresh.groupby(["budget", "max_steps"], dropna=False):
        metric_figure(group)

## 3. Corrected comparisons

Equal epochs are not equal updates when clip counts differ. The fixed-update grids explicitly hold updates constant within each comparison.

In [ ]:
epoch_runs = experiment_grid("mixture", seeds=(42, 43, 44))
small_controls = experiment_grid("updates", seeds=(42, 43, 44))
large_controls = experiment_grid("mixture_updates", seeds=(42, 43, 44))
display(plan(epoch_runs + small_controls + large_controls))
if RUN_TRAINING:
    execute(epoch_runs + small_controls + large_controls)

In [ ]:
display(corrected_results())